Article 8: Insurance & Property Analytics
Risk-Based Pricing Using Geo-Enriched Property & Hazard Zones on Databricks (Serverless Grid Indexing)

Insurance companies have traditionally relied on actuarial tables and zip code-level risk factors to price policies. But today's realities — wildfire seasons, climate-driven floods, urban crime pockets, hyper-local valuations — demand precise location-based intelligence.

With Databricks + spatial analytics (even without H3), you can run high-resolution property risk assessments by combining:

Property datasets (lat/lon, value, structure type)

Hazard zones (flood zones, wildfire polygons)

Crime heatmaps (police or open datasets)

Weather pattern data (storms, hail, tornadoes)

Mobility activity (for commercial properties)

This article walks through a real-world use case: calculating a Geospatial Risk Score for insurance underwriting — with Gold layer assets that feed pricing models and dashboards.

🧠 Business Use Case: Geo-Driven Home Insurance Pricing
💡 Objective:

Enable property insurers to price new policies by calculating a risk score using:

Risk Category	Data Input Required
Flood Risk	FEMA flood zones, river proximity
Wildfire Risk	Fire hazard index (polygon buffer zones)
Crime Risk	Geo-coded event density by cell
Weather Hazard	Hailstorm, hurricane zone polygons
Building Vulnerability	Construction type, age, elevation, etc.

Each risk category is evaluated per property using spatial joins with grid indexing.

The outcome? A unified Property Risk Score that helps:

✅ Underwriters approve or deny a claim
✅ Pricing teams set appropriate premiums
✅ Home owners learn about their risk profile
✅ Regulators track environmental equity

🗺️ Architecture Overview
[ Property Master ]
[ Hazard Zone Layers ]
[ Crime Density Map ]
[ Real Estate + Demographics ]
        |
        v
   BRONZE  (raw ingest)
        |
        v
   SILVER  (validate + geo + grid)
        |
        v
   GOLD    (risk metrics, risk_score, pricing tier)

👇 Step 1: Load Property Master (Bronze)

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS geo_demo;
USE CATALOG geo_demo;
CREATE SCHEMA IF NOT EXISTS insurance_bronze;
USE SCHEMA insurance_bronze;

CREATE SCHEMA IF NOT EXISTS insurance_silver;
USE SCHEMA insurance_silver;

CREATE SCHEMA IF NOT EXISTS insurance_gold;
USE SCHEMA insurance_gold;

In [0]:
from pyspark.sql.functions import expr, current_timestamp
from pyspark.sql.types import *
import random

# Mock 100 properties in Orlando, FL
data_properties = [
    (
        f"Prop_{i}",
        28.5383 + random.random() * 0.015,   # lat near Orlando
        -81.3792 + random.random() * 0.015,  # lon near Orlando
        random.randint(100000, 600000),      # value
        random.choice(["wood", "brick", "concrete"])
    )
    for i in range(100)
]

schema = StructType([
    StructField("property_id", StringType(), False),
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True),
    StructField("property_value", IntegerType(), True),
    StructField("structure_type", StringType(), True),
])

df_bronze = spark.createDataFrame(data_properties, schema) \
    .withColumn("ingest_ts", current_timestamp()) \
    .withColumn("src_file", expr("'property_batch'"))

df_bronze.write.format("delta").mode("overwrite").saveAsTable("insurance_bronze.properties")
df_bronze.show(5, truncate=False)


Stored in insurance_bronze.properties

### Step 2: Add Spatial Indexing to Properties (Silver)

In [0]:
%sql
CREATE OR REPLACE TABLE insurance_silver.properties AS
SELECT
  property_id,
  lat, lon,
  ST_POINT(lon, lat) AS geo_point,
  property_value,
  structure_type,
  CONCAT(FLOOR(lat * 100), '_', FLOOR(lon * 100)) AS grid_id,
  ingest_ts, src_file
FROM insurance_bronze.properties
WHERE lat BETWEEN -90 AND 90 AND lon BETWEEN -180 AND 180;


Now each property is tagged with a grid tile for fast spatial comparison.

###  Step 3: Load Flood Hazard Zones (Silver)

Assume a flood zone dataset exists, geo-bucketed using the same grid_id technique.

In [0]:
flood_zones = [
    ("FZ_1", "high", 28.5400, -81.3825),
    ("FZ_2", "medium", 28.5420, -81.3798),
    ("FZ_3", "low", 28.5386, -81.3753)
]

df_flood = spark.createDataFrame(flood_zones, ["zone_id", "intensity", "lat", "lon"]) \
    .withColumn("geo_point", expr("ST_POINT(lon, lat)")) \
    .withColumn("grid_id", expr("concat(floor(lat * 100), '_', floor(lon * 100))"))

df_flood.write.format("delta").mode("overwrite").saveAsTable("insurance_silver.flood_zones")


### Step 4: GOLD Layer – Calculate Property Risk Score

### A) Join Property with Hazard Zones via Approximate Grid Match

In [0]:
%sql
CREATE OR REPLACE TABLE insurance_gold.flood_matches AS
SELECT
  p.property_id,
  z.zone_id,
  z.intensity
FROM insurance_silver.properties p
JOIN insurance_silver.flood_zones z 
  ON p.grid_id = z.grid_id;


Fast lookup for flood correspondence — no expensive geometry needed first.

### B) Add Exact Spatial Refinement (Optional)

In [0]:
%sql
CREATE OR REPLACE TABLE insurance_gold.flood_risk AS
SELECT
  p.property_id,
  z.intensity,
  ST_DISTANCE(p.geo_point, z.geo_point) AS distance_m
FROM 
  insurance_silver.properties p
JOIN insurance_silver.flood_zones z
  ON p.grid_id = z.grid_id
WHERE ST_DISTANCE(p.geo_point, z.geo_point) <= 1000;  -- 1 km flood zone buffer


### C) Build Composite Risk Score

In [0]:
%sql
CREATE OR REPLACE TABLE insurance_gold.property_risk_score AS
SELECT
  p.property_id,
  p.property_value,
  COALESCE(MAX(CASE WHEN f.intensity = 'high' THEN 3
                    WHEN f.intensity = 'medium' THEN 2
                    WHEN f.intensity = 'low' THEN 1 END), 0) AS flood_score,
  CASE WHEN structure_type = 'wood' THEN 2 
       WHEN structure_type = 'brick' THEN 1 
       ELSE 0 END AS structure_score,
  -- final score = flood + structure
  (COALESCE(MAX(CASE WHEN f.intensity = 'high' THEN 3 
                     WHEN f.intensity = 'medium' THEN 2 
                     WHEN f.intensity = 'low' THEN 1 END), 0)
   +
   CASE WHEN structure_type = 'wood' THEN 2 
        WHEN structure_type = 'brick' THEN 1 
        ELSE 0 END
  ) AS risk_score
FROM insurance_silver.properties p
LEFT JOIN insurance_gold.flood_risk f
  ON p.property_id = f.property_id
GROUP BY p.property_id, p.property_value, p.structure_type;


In [0]:
%sql

select * from insurance_gold.property_risk_score

### Outputs a risk_score = flood_score + structure_score

![](/Volumes/thedataengineering_00/data/sampledata/8_BusinessBenefits.png)

Enhancements

- Join with crime dataset to build multi-factor risk
- Include storm path events for short-term pricing
- Build MLflow risk model to predict claim probability
- Export Gold tables to Power BI for interactive mapping

### Wrap-up

- You ingested raw property data → Bronze
- Spatially enabled and index optimized in Silver
- Built high-value Gold asset for risk-based policy pricing
- All done without H3, fully compatible with Databricks Serverless